In [ ]:
import sys
import logging
from pprint import pprint

from tqdm import tqdm
import pyarrow as pa

In [ ]:
sys.path.append("../")

from src.data import Document, BIERDataset
from src.chunking import Chunk, Chunker
from src.embeddings import Embedding, Embedder
from src.index import Indexer

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
dataset = BIERDataset()
chunker = Chunker()
embedder = Embedder()


path_qrel = "../datasets/beir/datasets/scifact/qrels/train.tsv"
path_corpus = "../datasets/beir/datasets/scifact/corpus.jsonl"
path_queries = "../datasets/beir/datasets/scifact/queries.jsonl"

In [ ]:
# Process documents
documents_loader = dataset.document_loader(file_path=path_corpus, batch_size=512)

indexer = Indexer(uri="lancedb_index", table_name="scifact_corpus", mode="append")

# # Stream documents in batches
# for idx, batch in tqdm(enumerate(documents_loader)):

#     # Chunk
#     chunks = chunker.fixed_size_chunking(documents=batch, chunk_size=5000)
#     logger.debug(f"chunking docs with ID: {[c.doc_id for c in chunks]}")


#     # Embedding
#     embeddings = embedder.embed_chunks(chunks)
#     logger.debug(f"embedding chunks with doc_id: {[e.chunk.doc_id for e in embeddings]}")


#     # Add to index
#     indexer.add_to_index(embeddings)
#     logger.debug(f'added batch {idx} to index')

In [ ]:
indexer.get_index().open_table("scifact_corpus").to_pandas().head()

In [ ]:
# Retrieval
retrievals = []
query_loader = dataset.query_loader(path_queries, batch_size=128)

# Stream queries in batches.
for _, batch in tqdm(enumerate(query_loader)):
    embeddings = embedder.embed_queries(batch)
    batch_results = indexer.search(embeddings, top_k=10)
    retrievals.append(batch_results)

retrievals = pa.concat_tables(retrievals)
retrievals_df = retrievals.to_pandas()

In [ ]:
# Evaluation
from src.evaluation import Evaluator

evaluator = Evaluator()

# Convert to pytrec_eval format
run = evaluator.df_to_pytrec(retrievals_df)

with open(path_qrel, "r") as f_qrel:
    qrel = evaluator.parse_qrel(f_qrel, ignore_header=True)

# evaluate
measures = [
    "map_cut.1",
    "ndcg_cut.1",
    "recall.1",
    "P.1",
    "map_cut.5",
    "ndcg_cut.5",
    "recall.5",
    "P.5",
    "map_cut.10",
    "ndcg_cut.10",
    "recall.10",
    "P.10",
]

results = evaluator.evaluate(qrel, run, measures)
logger.info("Evalutation completed!")